# BEZP Pretraining Wrapper (Google Colab - GPU)
This notebook acts as a thin wrapper to run the pretraining pipeline on a Google Colab GPU instance.

In [ ]:
# Verify GPU availability
!nvidia-smi

In [ ]:
# 1. Clone the repository
!git clone https://github.com/sidd-harth-r/Bandwidth-Equitable-Zero-Transmission-Proctoring.git
%cd Bandwidth-Equitable-Zero-Transmission-Proctoring

In [ ]:
# 2. Install dependencies
!pip install lightning torchmetrics h5py onnx onnx-tf tensorflowjs face-alignment

## 3. Columbia Gaze Extraction
Upload your Columbia Gaze dataset zip to Colab, unzip it, and run the extraction below.

In [ ]:
# Replace with your actual unzipped path
COLUMBIA_INPUT = '/content/columbia_gaze_data_set'
COLUMBIA_OUTPUT = '/content/drive/MyDrive/BEZP_Datasets/columbia_gaze_scores.npy'

!python ml/preprocessing/extract_columbia_colab.py --input $COLUMBIA_INPUT --output $COLUMBIA_OUTPUT --log-interval 100

## 4. Main Pretraining
Once all feature files (Columbia, MPIIGaze, DISFA, CMU) are in your Drive folder, run the training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Directory containing all .npy and .json extracted files
DATA_DIR = '/content/drive/MyDrive/BEZP_Datasets'

!python ml/models/lstm_anomaly_detector/pretrain.py --epochs 20 --batch-size 64 --data-dir $DATA_DIR --output-dir /content/exports

In [ ]:
# 5. Convert to TFJS
!onnx-tf convert -i /content/exports/bezp_model.onnx -o /content/exports/saved_model
!tensorflowjs_converter --input_format=tf_saved_model /content/exports/saved_model /content/exports/tfjs_model